# 📊 Amazon Sales — Exploratory Data Analysis (EDA)
**Source:** `Cleaned_Data.csv`  
**Tools:** Pandas, NumPy  
**Sections:** Descriptive Statistics · Correlation · Trend · Category · Segment · Pattern Detection

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('../Dataset/Cleaned_Data.csv', parse_dates=['order_date'])
print('Shape:', df.shape)
df.head(3)

## 1. Descriptive Statistics

In [ ]:
num = ['price','discount_percent','quantity_sold','rating','review_count',
        'discounted_price','total_revenue','profit','cost']
df[num].describe().T

In [ ]:
print('Date range :', df['order_date'].min().date(), '→', df['order_date'].max().date())
print('Orders     :', f"{df['order_id'].nunique():,}")
print('Products   :', f"{df['product_id'].nunique():,}")
print('Total Rev  :', f"${df['total_revenue'].sum():,.0f}")
print('Total Profit:', f"${df['profit'].sum():,.0f}")
print('AOV        :', f"${df['total_revenue'].mean():,.2f}")

## 2. Correlation Analysis

In [ ]:
corr = df[['price','discount_percent','quantity_sold','rating','review_count',
          'total_revenue','profit','cost']].corr()
plt.figure(figsize=(9,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=.5)
plt.title('Correlation Heatmap')
plt.tight_layout(); plt.show()
corr

**Finding:** `total_revenue` correlates strongly with `quantity_sold` (~0.59) and `price` (~0.55) — revenue is volume × price driven. `rating` shows ~0 correlation with revenue, and `discount_percent` is mildly negative (−0.14), i.e. deeper discounts do not lift revenue line-by-line.

## 3. Trend Analysis

In [ ]:
monthly = df.groupby('year_month')['total_revenue'].sum().reset_index()
plt.figure(figsize=(13,4))
plt.plot(monthly['year_month'], monthly['total_revenue'], marker='o', color='#146EB4')
plt.title('Monthly Revenue Trend (2022–2023)'); plt.xticks(rotation=45)
plt.ylabel('Revenue ($)'); plt.tight_layout(); plt.show()

In [ ]:
yoy = df.groupby('year').agg(Revenue=('total_revenue','sum'),
                          Profit=('profit','sum'),
                          Orders=('order_id','count'))
yoy['Growth %'] = yoy['Revenue'].pct_change()*100
yoy

## 4. Category Analysis

In [ ]:
cat = df.groupby('product_category').agg(Revenue=('total_revenue','sum'),
                                       Profit=('profit','sum'),
                                       Orders=('order_id','count'),
                                       Avg_Rating=('rating','mean')).sort_values('Revenue', ascending=False)
cat['Margin %'] = cat['Profit']/cat['Revenue']*100
cat

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,5))
cat['Revenue'].sort_values().plot(kind='barh', ax=ax[0], color='#FF9900')
ax[0].set_title('Revenue by Category'); ax[0].set_xlabel('$')
cat['Margin %'].sort_values().plot(kind='barh', ax=ax[1], color='#146EB4')
ax[1].set_title('Profit Margin % by Category'); ax[1].set_xlabel('%')
plt.tight_layout(); plt.show()

## 5. Segment Analysis (Region, Payment, Discount Band)

In [ ]:
seg = df.groupby('customer_region').agg(Revenue=('total_revenue','sum'),
                                        Profit=('profit','sum'),
                                        Orders=('order_id','count'),
                                        AOV=('total_revenue','mean')).sort_values('Revenue', ascending=False)
seg

In [ ]:
pay = df.groupby('payment_method').agg(Revenue=('total_revenue','sum'),
                                         Orders=('order_id','count')).sort_values('Revenue', ascending=False)
fig, ax = plt.subplots(1,2, figsize=(14,5))
pay['Orders'].plot(kind='pie', autopct='%1.1f%%', ax=ax[0], colors=sns.color_palette('Set2'))
ax[0].set_title('Order Share by Payment Method'); ax[0].set_ylabel('')
pay['Revenue'].sort_values().plot(kind='barh', ax=ax[1], color='#FF9900')
ax[1].set_title('Revenue by Payment Method'); ax[1].set_xlabel('$')
plt.tight_layout(); plt.show()

## 6. Pattern Detection

In [ ]:
dow = df.groupby('day_of_week')['total_revenue'].sum().reindex(
       ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
plt.figure(figsize=(10,4))
dow.plot(kind='bar', color='#146EB4'); plt.title('Revenue by Day of Week')
plt.ylabel('$'); plt.xticks(rotation=30); plt.tight_layout(); plt.show()
print('Weekend vs Weekday AOV:')
print(df.groupby('is_weekend')['total_revenue'].mean().round(2))

In [ ]:
discount_eff = df.groupby('discount_band', observed=True).agg(
    Orders=('order_id','count'), Revenue=('total_revenue','sum'),
    Avg_Rating=('rating','mean'), AOV=('total_revenue','mean')).round(2)
discount_eff

### 🧠 EDA Key Findings
1. **Revenue is flat YoY** (~+0.5%), with gentle seasonality peaking in Q3.
2. **Beauty** leads in revenue *and* margin (45%); **Home & Kitchen** is the lowest-margin category.
3. **Discounting is ineffective**: High-discount band has the *lowest* AOV; no rating lift.
4. **Regions are balanced** (Middle East & North America marginally top).
5. **Rating is uniformly ~3.0** — a clear customer-experience red flag.
6. Revenue scales with **quantity & price**, not with ratings or reviews.